In [49]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

There are two ways to impolement linear regression.

1. The closed solution using Ordinary Least Squares (OLS)
2. Computationally using Stochasitc Gradient Descent (SGD)

There is much theory here, but this is just focused on the implementation. Below are the ways to create linear regression

2
### 1. Analytically (OLS)

\begin{align}
\hat{\beta} = \left(X^TX\right)^{-1}X^TY
\end{align}

### 2. Numerically (SGD)

$$
\hat{\beta}^{(t+1)} = \hat{\beta}^{(t)} - \Delta\beta
$$

In [156]:
forestfires = pd.read_csv('../toydata/forest_fires/forestfires.csv')
X = forestfires.drop(['month', 'day', 'area'], axis=1)
y = forestfires[['area']]

### 1. Analytically

We do not need to scale our inputs as there are no variable interactions here. 

Also we don't need to center, but we could if we want to, then we would not need the intercept

In [157]:
# Implementing the formula
X['intercept'] = 1 # Don't forget to add the intercept
beta = np.linalg.inv(X.T @ X) @ X.T @ y

In [158]:
# Sklearn implementation
tmp = LinearRegression(fit_intercept=False, ).fit(X, y) # intercept is already added in dataframe
sklearn_coef = tmp.coef_.squeeze(0)

In [159]:
# Compare
pd.DataFrame(
    {
        'beta': beta.values.squeeze(1),
        'sklearn': sklearn_coef,
        'diff': np.abs(beta.values.squeeze(1) - sklearn_coef)
    }
)

,beta,sklearn,diff
0,1.907945,1.907945,1.223466e-13
1,0.569181,0.569181,7.793766e-14
2,-0.039200,-0.039200,5.627443e-15
3,0.077335,0.077335,4.857226e-16
4,-0.003295,-0.003295,1.214306e-16
5,-0.713739,-0.713739,2.775558e-15
6,0.800213,0.800213,6.994405e-15
7,-0.230645,-0.230645,8.326673e-17
8,1.557431,1.557431,7.327472e-15
9,-3.404037,-3.404037,2.264855e-14


The small difference is fine, sklearn uses a different method to solve for $\beta$


### 1. Analytically (OLS)


\begin{align}
LSE & = \mid\mid y - X\beta \mid\mid^2\\
& = (y - X\beta)^T(y - X\beta) \\
& = y^Ty - y^TX\beta - (X\beta)^Ty + (X\beta)^TX\beta \\
& = y^Ty - y^TX\beta - \beta^TX^Ty + \beta^TX^TX\beta \\ 
& = y^Ty - 2\beta^TX^Ty + \beta^TX^TX\beta \tag*{(since $y^TX\beta$ and $\beta^TX^Ty$ are the same scalars)}
\end{align}
Takeing the derivative w.r.t. $\beta$, setting equal to $0$, and solving, we get
\begin{align}
\frac{\partial LSE}{\partial\beta} & = \frac{\partial}{\partial\beta}\left(y^Ty - 2\beta^TX^Ty + \beta^TX^TX\beta\right)\\
& = -2X^Ty + 2X^TX\beta \\
0 & = -2X^Ty + 2X^TX\beta \\
X^Ty & = X^TX\beta \\
(X^TX)^{-1}X^Ty & = \beta \\
\end{align}
Resulting in our closed form solution:
\begin{align}
\hat{\beta} = \left(X^TX\right)^{-1}X^TY
\end{align}

### 2. Numerically (SGD)

$$
\hat{\beta}^{(t+1)} = \hat{\beta}^{(t)} - \Delta\beta
$$

Where $\Delta\beta$ is our gradient update step